# Capítulo 8: Machine Learning

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 11 de Grus (2019).

> Estou sempre pronto a aprender, embora nem sempre goste de ser ensinado.
>
> — Winston Churchill

Muita gente imagina que ciência de dados é basicamente aprendizado de máquina, e que cientistas de dados passam o dia construindo, treinando e ajustando modelos. Não é bem assim. Na prática, ciência de dados é principalmente transformar problemas de negócio em problemas de dados, coletar dados, entender dados, limpar dados e formatar dados — depois de tudo isso, o aprendizado de máquina é quase um detalhe final.

Ainda assim, é um detalhe interessante e essencial, e você precisa conhecê-lo para trabalhar com dados.

Este capítulo não constrói nenhum modelo. Ele constrói o **vocabulário** com que os oito capítulos seguintes vão ser julgados: o que significa um modelo estar bom, por que um modelo que acerta tudo pode ser inútil, e por que a resposta para "qual é a acurácia?" quase nunca é a pergunta certa.

> **❗ Importante**
>
> Se você leu o [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html), já viu dois modelos errados de formas instrutivas: uma média de salários que não significava nada e um classificador cujos cortes foram lidos dos próprios dados que ele deveria explicar.
>
> Este é o capítulo que dá nome aos dois problemas e ensina a detectá-los antes de publicar o resultado.

Ao final deste capítulo, você será capaz de:

- Definir o que é um modelo e o que distingue aprendizado de máquina de modelagem em geral
- Explicar o que são *overfitting* e *underfitting*, e por que o primeiro é invisível sem dados separados
- Dividir dados em treino e teste, e explicar por que uma divisão em duas partes às vezes não basta
- Construir uma matriz de confusão e calcular acurácia, precisão, revocação e F1
- Argumentar por que acurácia sozinha é uma métrica enganosa em problemas desbalanceados
- Diagnosticar um modelo ruim como problema de viés ou de variância, e saber que ação cada diagnóstico sugere
- Reconhecer o que é um atributo (*feature*) e como a escolha deles restringe os modelos disponíveis

## Seções

| Seção | Tópico |
|---|---|
| [8.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/01-modelagem.html) | Modelagem |
| [8.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/02-o-que-e-machine-learning.html) | O que é Machine Learning? |
| [8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) | Overfitting e Underfitting |
| [8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html) | Correção |
| [8.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-vies-e-variancia.html) | O Compromisso Viés-Variância |
| [8.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html) | Extração e Seleção de Atributos |

## Modelagem

> **📌 Nota**
>
> Esta seção corresponde a *Modeling*, do capítulo 11 de Grus (2019).

Antes de falar de aprendizado de máquina, é preciso falar de **modelos**.

O que é um modelo? É simplesmente a especificação de uma relação matemática — ou probabilística — que existe entre variáveis diferentes.

> **🔷 Conceito**
>
> Um **modelo** é uma afirmação sobre como grandezas se relacionam.
>
> Ele não precisa ser complicado, não precisa envolver estatística, e não precisa ser aprendido de dado nenhum. "O preço final é o preço da unidade vezes a quantidade" já é um modelo.

### Modelos que você já usa

Se você quisesse levantar dinheiro para o seu site de rede social, provavelmente montaria um **modelo de negócio** — quase certamente numa planilha — que recebe entradas como "número de usuários", "receita de anúncio por usuário" e "número de funcionários", e devolve o lucro anual projetado para os próximos anos.

Uma **receita de cozinha** também é um modelo: ela relaciona entradas como "número de pessoas" e "quanta fome" a quantidades de cada ingrediente.

E se você já assistiu a pôquer na televisão, sabe que a "probabilidade de vitória" de cada jogador é estimada em tempo real por um modelo que leva em conta as cartas já reveladas e a distribuição de cartas no baralho.

### De onde vem cada modelo

Repare que os três exemplos acima têm origens completamente diferentes — e essa diferença é o ponto desta seção.

**O modelo de negócio** se apoia em relações matemáticas simples e conhecidas de antemão: lucro é receita menos despesa, receita é unidades vendidas vezes preço médio, e assim por diante. Ninguém precisou de dado para descobrir isso; a estrutura vem da contabilidade.

**O modelo da receita** provavelmente veio de tentativa e erro. Alguém foi para a cozinha, testou combinações de ingredientes até achar uma que funcionasse, e escreveu o resultado. É empírico, mas o "ajuste" foi feito por uma pessoa provando a comida.

**O modelo do pôquer** se apoia em teoria da probabilidade, nas regras do jogo, e em algumas hipóteses razoavelmente inocentes sobre o processo aleatório de distribuir cartas. A estrutura vem da matemática, não dos dados de partidas anteriores.

> **🟩 Exemplo**
>
> Nenhum desses três é aprendizado de máquina.
>
> Todos são modelos legítimos, todos podem ser úteis, e nenhum deles aprendeu coisa alguma a partir de dados. Isso importa porque a distinção que vem a seguir — o que é aprendizado de máquina — só faz sentido depois de reconhecer que **modelar é mais amplo que aprender**.
>
> E vale a inversão: quando alguém apresenta um modelo, uma pergunta produtiva é *de onde veio a estrutura dele*. Da teoria? Do dado? Do palpite de quem o montou? As três respostas levam a formas diferentes de duvidar do resultado.

## O que é Machine Learning?

> **📌 Nota**
>
> Esta seção corresponde a *What Is Machine Learning?*, do capítulo 11 de Grus (2019).

Cada pessoa tem a sua definição exata, mas vamos usar **aprendizado de máquina** para nos referir a criar e usar modelos que são *aprendidos a partir de dados*. Em outros contextos isso poderia se chamar *modelagem preditiva* ou *mineração de dados*; aqui ficamos com aprendizado de máquina.

> **🔷 Conceito**
>
> A diferença em relação à seção anterior é exatamente uma: **os parâmetros do modelo vêm do dado, não de quem o escreveu.**
>
> No modelo de negócio, alguém digitou a margem de lucro. Num modelo aprendido, você especifica a *forma* — uma reta, uma árvore, uma rede — e o dado decide os números que a preenchem.

Tipicamente, o objetivo é usar dados existentes para desenvolver modelos que possam *prever* resultados para dados novos, como:

- Se uma mensagem de e-mail é spam ou não
- Se uma transação de cartão de crédito é fraudulenta
- Em qual anúncio um comprador tem mais chance de clicar
- Qual time vai ganhar o campeonato

### Supervisionado e não supervisionado

Vamos ver dois tipos de modelo neste livro.

**Supervisionados** são aqueles em que existe um conjunto de dados rotulado com as respostas corretas, das quais o modelo aprende. É o caso da maioria dos capítulos: k-vizinhos, Naive Bayes, regressão, árvores de decisão e redes neurais são todos supervisionados.

**Não supervisionados** são aqueles em que não há rótulo algum. O [Capítulo 17](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html), sobre clustering, é o exemplo deste livro: os dados chegam sem categoria e o algoritmo precisa descobrir grupos sozinho.

Existem outros tipos que este livro não cobre, mas vale conhecer os nomes:

- **Semissupervisionado**, em que apenas parte dos dados tem rótulo — situação comum na prática, porque rotular custa caro.
- **Online**, em que o modelo se ajusta continuamente conforme dados novos chegam, em vez de ser treinado uma vez.
- **Por reforço**, em que o modelo faz uma série de previsões e depois recebe um sinal indicando o quão bem se saiu.

> **🟩 Exemplo**
>
> A distinção entre supervisionado e não supervisionado costuma ser apresentada como técnica, mas ela é antes de tudo **econômica**.
>
> Rótulo é caro. Alguém precisa olhar cada e-mail e dizer "spam" ou "não spam", examinar cada transação e dizer "fraude" ou "legítima", ler cada exame e dar um diagnóstico. Em muitos problemas reais, o dado bruto é abundante e o rótulo é o gargalo.
>
> É por isso que os métodos não supervisionados continuam relevantes mesmo sendo, em geral, menos precisos: eles trabalham com o dado que se tem de graça.

### Famílias parametrizadas

Mesmo na situação mais simples, existe um universo inteiro de modelos que poderiam descrever a relação que nos interessa. Na maior parte dos casos, nós mesmos escolhemos uma **família parametrizada** de modelos e usamos os dados para aprender parâmetros que sejam, de algum modo, ótimos.

Por exemplo: podemos supor que a altura de uma pessoa é, aproximadamente, uma função linear do peso dela — e então usar dados para aprender qual é essa função linear. Ou podemos supor que uma árvore de decisão é um bom jeito de diagnosticar doenças, e usar dados para aprender qual é a árvore "ótima".

> **❗ Importante — A escolha da família é sua, e ela não é neutra**
>
> Repare no que essa formulação esconde. Quando você decide "vou ajustar uma reta", já decidiu que a relação é linear — e nenhum dado vai contradizer isso, porque o ajuste só escolhe *qual* reta, nunca *se* deveria ser uma reta.
>
> Um modelo mal escolhido não falha com erro. Ele devolve os melhores parâmetros possíveis para uma forma errada, e o resultado parece um resultado.
>
> Boa parte deste capítulo é sobre como perceber isso a tempo.

O resto do livro investiga famílias diferentes de modelos que podemos aprender. Mas antes disso é preciso entender melhor os fundamentos — e é o que as próximas quatro seções fazem.

## Overfitting e Underfitting

> **📌 Nota**
>
> Esta seção corresponde a *Overfitting and Underfitting*, do capítulo 11 de Grus (2019).

Um perigo comum em aprendizado de máquina é o **overfitting** — produzir um modelo que se sai bem nos dados em que foi treinado, mas generaliza mal para dados novos. Isso pode envolver aprender *ruído* nos dados, ou aprender a identificar as entradas específicas em vez dos fatores que de fato predizem o resultado desejado.

O outro lado da moeda é o **underfitting** — produzir um modelo que não vai bem nem nos dados de treino, embora nesse caso a reação usual seja concluir que o modelo não presta e continuar procurando outro.

O gráfico abaixo ajusta três polinômios a uma amostra de dados. Não se preocupe com *como* eles foram ajustados; isso vem em capítulos posteriores.

In [ ]:
# Figura: Overfitting e underfitting: polinômios de grau 0, 1 e 9 ajustados aos mesmos dez pontos
import numpy as np
from matplotlib import pyplot as plt

rng = np.random.default_rng(12)

# Dez pontos ao redor da reta y = 2x
xs = 2.5 + 0.65 * np.arange(10)
ys = 2 * xs + rng.normal(0, 1.2, size=xs.shape)

# --- grau 0: a média ---
media = ys.mean()

# --- grau 1: mínimos quadrados, em duas fórmulas ---
dx = xs - xs.mean()
dy = ys - ys.mean()
slope = (dx * dy).sum() / (dx ** 2).sum()
intercept = ys.mean() - slope * xs.mean()

# --- grau 9: o polinômio que passa exatamente pelos 10 pontos ---
# Com n pontos distintos existe um único polinômio de grau n-1 que passa por
# todos eles, e a forma de Lagrange o escreve diretamente — sem resolver sistema.
# O laço percorre os n termos da soma, que é a fórmula; o produto sobre os
# outros pontos e a avaliação em toda a grade são operações de array.
def lagrange(x, xs, ys):
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for i in range(len(xs)):
        outros = np.arange(len(xs)) != i           # máscara: todos menos o i
        # x[:, None] põe a grade em coluna (401, 1) para casar com xs[outros] (9,)
        base = np.prod((x[:, None] - xs[outros]) / (xs[i] - xs[outros]), axis=1)
        total += ys[i] * base
    return total

grade = np.linspace(xs[0], xs[-1], 401)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(grade, np.full_like(grade, media), "--", label="grau 0", color="#4472c4")
ax.plot(grade, intercept + slope * grade, "-", label="grau 1", color="#2e7d32")
ax.plot(grade, lagrange(grade, xs, ys), ":", label="grau 9", color="#c00000")
ax.scatter(xs, ys, color="#1f3864", zorder=5)

ax.set_ylim(ys.min() - 6, ys.max() + 6)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Melhores polinômios de vários graus")
ax.legend()
plt.tight_layout()
plt.show()

A linha horizontal é o melhor polinômio de grau 0 — uma constante. Ela sofre de **underfitting** severo: não acompanha nada.

O polinômio de grau 9, com dez parâmetros, passa **exatamente** por cada ponto de treino. E sofre de **overfitting** igualmente severo: se pegássemos alguns pontos a mais da mesma população, ele erraria por muito.

Repare no formato dele. Entre os pontos, ele sobe e desce em ondas que não correspondem a nada — sobe a 10 entre o segundo e o terceiro pontos, que valem 7,6 e 8,5, e a 13,3 entre o sétimo e o oitavo. Nas duas pontas ele nem cabe na figura: mergulha abaixo de zero logo depois do primeiro ponto e dispara acima de 28 antes do último. Essas ondulações são o preço de ser obrigado a passar por todos os pontos: como o ruído de cada observação é diferente, a curva precisa se contorcer para alcançá-los um a um. **Ela está ajustando o ruído**, e ruído não se repete na próxima amostra.

(Na [seção sobre viés e variância](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/05-vies-e-variancia.html) você vai ver o que acontece quando se repete este ajuste com dez amostras diferentes: as curvas discordam entre si em toda a faixa, e quatro delas ainda escapam da moldura nas pontas — todas por baixo, logo depois do primeiro ponto, e duas também por cima, antes do último. É essa discordância que dá nome à variância.)

A reta de grau 1 acha um bom equilíbrio: fica razoavelmente perto de todos os pontos, e se estes dados forem representativos, ela provavelmente vai ficar perto de pontos novos também.

> **🔷 Conceito**
>
> O polinômio de grau 9 tem **erro zero** nos dados de treino. Ele é, por essa medida, o melhor dos três — e é o pior dos três.
>
> Guarde esta frase: **erro baixo nos dados de treino não é evidência de nada.** É por isso que a próxima parte desta seção existe.

### Separar treino e teste

Modelos complexos demais levam a overfitting e não generalizam além dos dados em que foram treinados. Como garantir que os nossos modelos não são complexos demais?

A abordagem mais fundamental é **usar dados diferentes para treinar e para testar o modelo.**

A forma mais simples de fazer isso é dividir o conjunto de dados, usando (por exemplo) dois terços para treinar e medindo o desempenho no terço restante:

In [ ]:
from typing import Tuple

import numpy as np

def split_data(data, prob: float, rng: np.random.Generator):
    """Divide `data` nas frações [prob, 1 - prob], em ordem sorteada.

    Array -> dois arrays, fatiados pelas mesmas linhas.
    Sequência qualquer (uma lista de mensagens) -> duas listas.
    """
    idx = rng.permutation(len(data))       # uma permutação dos índices...
    cut = int(len(data) * prob)            # ...cortada na proporção pedida
    if isinstance(data, np.ndarray):
        return data[idx[:cut]], data[idx[cut:]]
    return [data[i] for i in idx[:cut]], [data[i] for i in idx[cut:]]

data = np.arange(1000)
train, test = split_data(data, 0.75, np.random.default_rng(0))

# As proporções devem estar corretas
assert len(train) == 750
assert len(test) == 250

# E os dados originais devem estar preservados (em alguma ordem)
assert np.array_equal(np.sort(np.concatenate([train, test])), data)

# O outro ramo: uma sequência qualquer devolve duas listas
mensagens = list(range(1000))
train_msg, test_msg = split_data(mensagens, 0.75, np.random.default_rng(0))
assert len(train_msg) == 750
assert sorted(train_msg + test_msg) == mensagens

len(train), len(test)

O `isinstance` no meio da função existe porque nem todo conjunto de dados é um array. Quando `data` é uma matriz de medidas — as flores do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html), as linhas de uma regressão —, `data[idx[:cut]]` é indexação por array: o `numpy` devolve de uma vez as linhas escolhidas, na ordem escolhida. Quando `data` é uma sequência de objetos que não cabem num array de números — a lista de mensagens de e-mail do [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html) —, a compreensão de lista faz o mesmo trabalho, índice a índice. As duas linhas dizem a mesma coisa em duas linguagens, e é por isso que um único `split_data` serve aos dois capítulos.

> **📌 Nota**
>
> Repare no que a função **não** faz: ela não embaralha `data`. `rng.permutation(len(data))` devolve um array com os índices `0, 1, …, n-1` em ordem sorteada, e tanto `data[idx[:cut]]` quanto a compreensão de lista **copiam** as posições escolhidas para uma coleção nova. O que o chamador passou continua exatamente como estava.
>
> A versão em listas de Grus (2019) precisa de uma cópia explícita antes de embaralhar, porque o `shuffle` da biblioteca padrão embaralha no lugar — e esquecer essa cópia é o tipo de efeito colateral que não gera erro e aparece muito depois, quando alguém descobre que a ordem original dos dados sumiu. Aqui a cópia é consequência da indexação, não uma linha que alguém precisa lembrar de escrever. O `assert` do fim — os mil números de volta, ordenados — é o que prova que nada foi perdido nem duplicado.

> **📌 Nota — A semente não é opcional**
>
> `split_data` recebe o gerador como parâmetro **obrigatório**: não há valor padrão, e nenhum estado global é consultado. Quem chama precisa escrever `np.random.default_rng(0)`, e a semente fica visível ao lado da divisão que ela produziu.
>
> Sem isso, "o modelo acertou 93% no teste" seria uma afirmação sobre um sorteio que ninguém consegue repetir — nem você, na semana seguinte. Todo capítulo daqui em diante divide dados assim, com o gerador escrito no chunk.

Muitas vezes teremos variáveis de entrada e de saída pareadas. Nesse caso é preciso garantir que os valores correspondentes fiquem juntos, seja no treino, seja no teste:

In [ ]:
def train_test_split(xs: np.ndarray,
                     ys: np.ndarray,
                     test_pct: float,
                     rng: np.random.Generator
                     ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """-> (x_train, x_test, y_train, y_test), fatiados pelos MESMOS índices."""
    xs = np.asarray(xs)
    ys = np.asarray(ys)
    assert len(xs) == len(ys), "xs e ys precisam ter o mesmo número de linhas"
    train_idx, test_idx = split_data(np.arange(len(xs)), 1 - test_pct, rng)
    return xs[train_idx], xs[test_idx], ys[train_idx], ys[test_idx]

> **🟩 Exemplo**
>
> A função não embaralha `xs` e `ys` separadamente — ela embaralha os **índices** e usa os mesmos índices nos dois arrays. `xs[train_idx]` é indexação por array: `train_idx` diz quais linhas, e em que ordem.
>
> Se embaralhasse cada uma por conta própria, cada entrada acabaria pareada com a saída de outra observação, e o modelo aprenderia a partir de pares aleatórios. Ele treinaria sem erro algum e produziria lixo.

Como sempre, queremos garantir que o código funciona:

In [ ]:
xs = np.arange(1000)           # xs são 0 ... 999
ys = 2 * xs                    # cada y_i é o dobro de x_i
x_train, x_test, y_train, y_test = train_test_split(xs, ys, 0.25, np.random.default_rng(0))

# Verifica que as proporções estão corretas
assert len(x_train) == len(y_train) == 750
assert len(x_test) == len(y_test) == 250

# Verifica que os pontos correspondentes estão pareados corretamente
assert np.all(y_train == 2 * x_train)
assert np.all(y_test == 2 * x_test)

len(x_train), len(x_test)

As duas funções ficam em `scratch_np.machine_learning`, exatamente como você as escreveu aqui — o módulo é o chunk salvo em arquivo. O [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html) importa `train_test_split` para dividir as flores do Iris — a matriz de medidas e o array de espécies, pelos mesmos índices —, o [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html) importa `split_data` para separar a lista de mensagens de e-mail, e o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) volta ao `train_test_split`. Nenhum deles reescreve a divisão.

Depois disso, você pode fazer algo assim:

```python
model = SomeKindOfModel()
x_train, x_test, y_train, y_test = train_test_split(xs, ys, 0.33, np.random.default_rng(0))
model.train(x_train, y_train)
performance = model.test(x_test, y_test)
```

Se o modelo tiver sofrido overfitting nos dados de treino, ele vai — espera-se — se sair muito mal nos dados de teste, que são completamente separados. Dito de outro jeito: se ele vai bem no teste, você pode ter mais confiança de que está *ajustando* em vez de *superajustando*.

### Dois jeitos de isso dar errado

Há, no entanto, dois modos de essa estratégia falhar.

**O primeiro** é haver padrões comuns entre treino e teste que não generalizariam para um conjunto maior.

Por exemplo: imagine que seus dados são de atividade de usuários, com uma linha por usuário por semana. Nesse caso, a maioria dos usuários vai aparecer tanto no treino quanto no teste, e certos modelos podem aprender a **identificar usuários** em vez de descobrir relações entre atributos. O Grus (2019) registra que isso já aconteceu com ele.

**O segundo é maior**, e é o que mais aparece na prática: usar a divisão treino/teste não apenas para *julgar* um modelo, mas para *escolher* entre vários.

> **⚠️ Atenção — O conjunto de teste vira treino quando você o usa para escolher**
>
> Suponha que você treine vinte modelos, meça todos no conjunto de teste e fique com o melhor.
>
> Cada modelo individualmente pode não estar superajustado. Mas "escolher o modelo que vai melhor no teste" é, ele próprio, um **meta-treinamento** — e o conjunto de teste acabou de virar um segundo conjunto de treino.
>
> O resultado é previsível: o modelo que foi melhor no teste vai ser bom no teste. Isso deixou de ser informação.
>
> A saída é dividir em **três** partes: um conjunto de **treino** para construir modelos, um de **validação** para escolher entre modelos treinados, e um de **teste** para julgar o modelo final — usado uma única vez, no fim.

Essa armadilha é sutil porque cada passo isolado parece correto. Ninguém treinou no conjunto de teste; só olhou para ele algumas vezes. Mas olhar e decidir é uma forma de aprender — e a informação vaza pela decisão de quem olha, não pelo código.

> **💡 Dica — Na prática: `scikit-learn`**
>
> A função que você acabou de escrever existe pronta, com o mesmo nome:
>
> ```python
> from sklearn.model_selection import train_test_split
>
> X_treino, X_teste, y_treino, y_teste = train_test_split(
>     xs, ys, test_size=0.25, random_state=42
> )
> ```
>
> A assinatura é `train_test_split(*arrays, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)`. Ela aceita quantos arrays você quiser e divide todos pelos mesmos índices — é o mesmo cuidado do nosso `train_test_split`, generalizado. Sem `test_size` nem `train_size`, o padrão é 25% para teste.
>
> **A primeira coisa que ela esconde é a semente — e vale dizer com cuidado o que "esconde" significa.** O nosso `split_data` recebe o gerador como parâmetro **obrigatório**: sem um `np.random.default_rng(42)` escrito na chamada, ele nem roda. O `train_test_split` da biblioteca tem `random_state=None` por padrão e, nesse caso, sorteia do gerador **global** do `numpy`: duas chamadas seguidas devolvem conjuntos diferentes, e criar um `np.random.default_rng(42)` antes não muda nada, porque o gerador que ela usa é outro. Só `random_state=42` torna a divisão reprodutível, e a regra vale para o pacote inteiro — todo objeto do `scikit-learn` que sorteia tem o seu próprio `random_state`. Os dois caminhos chegam ao mesmo lugar; a diferença é o padrão, e o dela deixa esquecer. O que ela esconde mesmo é a **permutação**: o nosso `split_data` monta `idx = rng.permutation(len(data))` e divide esse array, que dá para imprimir, guardar e conferir; o `train_test_split` devolve os pedaços e joga os índices fora. Para saber *quais* linhas foram para o teste — casar com um identificador, auditar uma divisão que deu resultado estranho —, o jeito é dividir um `np.arange(len(X))` junto com os dados, como se fosse mais uma variável, ou pedir os índices a `ShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X)`.
>
> **A segunda é o embaralhamento.** `shuffle=True` é o padrão, e é o que o nosso `split_data` também faz — mas a família de validação cruzada não segue essa regra. `KFold(n_splits=5)` tem `shuffle=False` por padrão: ele corta a lista em cinco pedaços contíguos, na ordem em que os dados chegaram. Sobre um arquivo ordenado por classe, isso é catastrófico. O `dados/iris.data` do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) é exatamente assim — as 50 *setosa* primeiro, depois as 50 *versicolor*, depois as 50 *virginica*:
>
> ```python
> from sklearn.model_selection import cross_val_score, KFold
> from sklearn.neighbors import KNeighborsClassifier
>
> cross_val_score(KNeighborsClassifier(3), X, y, cv=KFold(3))  # [0.   0.   0.  ]
> cross_val_score(KNeighborsClassifier(3), X, y, cv=3)         # [0.98 0.96 0.98]
> ```
>
> Zero acerto nas três dobras. Cada dobra de teste contém uma espécie inteira, e o treino não contém nenhum exemplar dela — o modelo é avaliado sobre uma classe que nunca viu. A segunda linha funciona porque `cv=3`, sem objeto explícito, faz o `scikit-learn` escolher `StratifiedKFold` sozinho quando o estimador é um classificador.
>
> **A terceira é a estratificação, e o padrão é inconsistente entre as duas funções.** `cross_val_score` estratifica por padrão; `train_test_split` **não** — `stratify=None`. Numa base com 8 positivos em 200 e 25% de teste, a contagem de positivos no conjunto de teste varia com a semente: em 200 sementes testadas, ela foi de 0 a 6, e **em 18 delas o conjunto de teste não tinha positivo nenhum**. Com `stratify=y`, foi exatamente 2 em todas. Um conjunto de teste sem positivos torna a revocação indefinida — é o assunto da [seção 8.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/04-correcao.html). Passar `stratify=y` junto com `shuffle=False` levanta `ValueError: Stratified train/test split is not implemented for shuffle=False`.
>
> **E a divisão em três partes, que esta seção defendeu, tem uma ferramenta própria** — que é também a armadilha do callout acima, automatizada. `GridSearchCV(estimator, param_grid, cv=None, refit=True)` treina um modelo para cada combinação de hiperparâmetros e guarda a melhor em `best_params_` e `best_score_`. Ele automatiza a etapa de **validação**: cada candidato é pontuado em dobras que não entraram no treino dele, então a nota de cada modelo, isolada, é honesta.
>
> O que ele **não** faz é o teste. E por duas razões distintas, que vale separar:
>
> - `best_score_` é a nota do **vencedor de uma competição**, não a nota de um modelo qualquer. Escolher o máximo entre vinte notas ruidosas devolve um número enviesado para cima — é exatamente o que o callout acima descreve, com o conjunto de validação no lugar do de teste.
> - `refit=True` é o padrão, e ele **retreina o vencedor sobre todos os dados que você passou ao `.fit()`** — inclusive os que serviram de validação. O `best_estimator_` que volta, portanto, já viu tudo o que você entregou.
>
> Nos dois casos o remédio é o mesmo, e é manual: separe o conjunto de teste com um `train_test_split` *antes* de o `GridSearchCV` existir, entregue a ele apenas a parte de treino, e não olhe para o teste até o fim.

## Correção

> **📌 Nota**
>
> Esta seção corresponde a *Correctness*, do capítulo 11 de Grus (2019).

Grus (2019) abre esta seção com uma piada que vale reproduzir inteira, porque ela é o melhor argumento do capítulo.

Nas horas vagas, conta ele, desenvolveu um teste barato e não invasivo, aplicável a recém-nascidos, que prevê — **com mais de 98% de acurácia** — se o bebê vai desenvolver leucemia. O advogado dele o convenceu de que o teste não é patenteável, então ele compartilha os detalhes:

> Preveja leucemia se, e somente se, o bebê se chamar Luke.

E o teste é, de fato, mais de 98% acurado. Também é um teste estupidamente ruim — e é uma boa ilustração de por que normalmente não se usa "acurácia" para medir a qualidade de um modelo de classificação binária.

### As quatro caixas

Imagine construir um modelo para tomar uma decisão **binária**. Este e-mail é spam? Devemos contratar esta candidata? Este passageiro é secretamente um terrorista?

Dado um conjunto de dados rotulado e um modelo preditivo desses, cada observação cai em uma de quatro categorias:

**Verdadeiro positivo** — "Esta mensagem é spam, e previmos corretamente que era spam."

**Falso positivo (erro Tipo 1)** — "Esta mensagem não é spam, mas previmos que era."

**Falso negativo (erro Tipo 2)** — "Esta mensagem é spam, mas previmos que não era."

**Verdadeiro negativo** — "Esta mensagem não é spam, e previmos corretamente que não era."

Frequentemente representamos essas contagens numa **matriz de confusão**:

|  | É spam | Não é spam |
|---|---|---|
| **Previu "spam"** | Verdadeiro positivo | Falso positivo |
| **Previu "não é spam"** | Falso negativo | Verdadeiro negativo |

### O teste do Luke, em números

Vamos ver como o teste da leucemia se encaixa nesse quadro. Hoje em dia, aproximadamente **5 bebês em cada 1.000 se chamam Luke**. E a prevalência da leucemia ao longo da vida é de cerca de 1,4%, ou **14 em cada 1.000 pessoas**.

Se acreditarmos que esses dois fatores são independentes e aplicarmos o teste "Luke é sinal de leucemia" a um milhão de pessoas, esperaríamos uma matriz de confusão assim:

|  | Leucemia | Sem leucemia | Total |
|---|---|---|---|
| **"Luke"** | 70 | 4.930 | 5.000 |
| **Não "Luke"** | 13.930 | 981.070 | 995.000 |
| **Total** | 14.000 | 986.000 | 1.000.000 |

Com esses números dá para calcular várias estatísticas de desempenho. A **acurácia** é definida como a fração de previsões corretas:

In [ ]:
def accuracy(tp: int, fp: int, fn: int, tn: int) -> float:
    correct = tp + tn
    total = tp + fp + fn + tn
    return correct / total

assert accuracy(70, 4930, 13930, 981070) == 0.98114

accuracy(70, 4930, 13930, 981070)

98,1%. Parece um número bastante impressionante.

> **⚠️ Atenção — De onde vem a acurácia de 98%**
>
> O teste acerta em 98,1% dos casos porque **quase ninguém tem leucemia e quase ninguém se chama Luke**. Os 981.070 verdadeiros negativos — pessoas que não se chamam Luke e não têm leucemia — carregam sozinhos a métrica inteira.
>
> Um teste que simplesmente respondesse "não tem leucemia" para todo mundo teria acurácia de 98,6%, ainda melhor. E não olharia para dado nenhum.
>
> Sempre que a classe de interesse é rara, a acurácia mede principalmente o tamanho da classe majoritária. Fraude, doença rara, defeito de fabricação, evasão escolar — todos esses são problemas em que reportar acurácia é, na prática, esconder o resultado.

### Precisão e revocação

Por isso é comum olhar a combinação de **precisão** e **revocação** (*recall*).

A precisão mede o quanto as nossas previsões *positivas* foram acertadas:

In [ ]:
def precision(tp: int, fp: int, fn: int, tn: int) -> float:
    return tp / (tp + fp)

assert precision(70, 4930, 13930, 981070) == 0.014

precision(70, 4930, 13930, 981070)

E a revocação mede que fração dos positivos o modelo conseguiu identificar:

In [ ]:
def recall(tp: int, fp: int, fn: int, tn: int) -> float:
    return tp / (tp + fn)

assert recall(70, 4930, 13930, 981070) == 0.005

recall(70, 4930, 13930, 981070)

Os dois números são péssimos, o que reflete que este é um modelo péssimo.

> **🔷 Conceito**
>
> Compare as três medidas do mesmo modelo:
>
> - **Acurácia: 98,1%** — parece excelente
> - **Precisão: 1,4%** — de cada 100 pessoas que o teste aponta, 1 ou 2 realmente têm a doença
> - **Revocação: 0,5%** — de cada 200 pessoas que têm a doença, o teste encontra 1
>
> É o mesmo modelo, nos mesmos dados. A diferença é que acurácia conta os acertos fáceis, e precisão e revocação contam apenas o que acontece na classe que interessa.

Às vezes precisão e revocação são combinadas no **F1**, definido como:

In [ ]:
def f1_score(tp: int, fp: int, fn: int, tn: int) -> float:
    p = precision(tp, fp, fn, tn)
    r = recall(tp, fp, fn, tn)
    return 2 * p * r / (p + r)

f1_score(70, 4930, 13930, 981070)

Este é a **média harmônica** de precisão e revocação, e necessariamente fica entre as duas.

As quatro funções desta seção moram em `scratch_np.machine_learning`, junto com o `split_data` da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) — o mesmo código que você acabou de escrever. É de lá que o [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html) e o [Capítulo 13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html) as importam para avaliar os modelos deles.

> **📌 Nota**
>
> Por que média harmônica e não média aritmética? Porque a harmônica é puxada para baixo pelo menor dos dois valores.
>
> Um modelo com precisão 1,0 e revocação 0,0 tem média aritmética 0,5 — parece mediano. A média harmônica dá 0. E 0 é a avaliação correta de um modelo que nunca encontra nada.

### O compromisso entre precisão e revocação

Normalmente, a escolha de um modelo envolve um compromisso entre precisão e revocação.

Um modelo que responde "sim" mesmo quando está pouco confiante provavelmente terá **revocação alta e precisão baixa** — encontra quase todos os casos, mas acusa muita gente à toa. Um modelo que só responde "sim" quando está extremamente confiante terá **precisão alta e revocação baixa** — quase nunca erra ao acusar, mas deixa passar a maioria.

Também dá para pensar nisso como um compromisso entre falsos positivos e falsos negativos: dizer "sim" com frequência demais produz muitos falsos positivos; dizer "não" com frequência demais produz muitos falsos negativos.

Imagine que existissem 10 fatores de risco para leucemia, e que quanto mais deles alguém tivesse, maior a chance de desenvolver a doença. Nesse caso dá para imaginar um contínuo de testes: "preveja leucemia se houver ao menos um fator de risco", "se houver ao menos dois", e assim por diante. Conforme você aumenta o limiar, aumenta a precisão do teste — porque quem tem mais fatores de risco tem mais chance de adoecer — e diminui a revocação, porque cada vez menos dos futuros doentes atingem o limiar. Escolher o limiar certo é achar o equilíbrio certo.

> **🟩 Exemplo**
>
> E "o equilíbrio certo" **não é uma pergunta técnica**. Depende do custo de cada erro, e esse custo vem do domínio.
>
> Num teste de triagem para uma doença tratável, um falso negativo é uma pessoa que não recebe tratamento; um falso positivo é um exame a mais. Os dois custos não se parecem, e o limiar deve pender para a revocação.
>
> Num filtro de spam, um falso positivo é um e-mail importante que some na caixa de lixo; um falso negativo é um anúncio na caixa de entrada. Aqui o desequilíbrio aponta para o outro lado, e o limiar deve pender para a precisão.
>
> Mesmo algoritmo, mesmas métricas, decisões opostas. Quem escolhe o limiar está fazendo uma escolha sobre quem paga o preço do erro — e vale saber que está fazendo isso.

> **💡 Dica — Na prática: `scikit-learn`**
>
> As métricas desta seção estão todas em `sklearn.metrics`, e recebem os rótulos em vez das quatro contagens:
>
> ```python
> from sklearn.metrics import (accuracy_score, precision_score, recall_score,
>                              f1_score, confusion_matrix, classification_report)
>
> accuracy_score(y_teste, previsto)    # 0.98114
> precision_score(y_teste, previsto)   # 0.014
> recall_score(y_teste, previsto)      # 0.005
> f1_score(y_teste, previsto)          # 0.007368421052631579
> ```
>
> Reconstruindo o milhão de pessoas do teste do Luke, os quatro números saem idênticos aos que você calculou acima. Não há truque nenhum nessas funções — elas contam as mesmas quatro caixas. O que muda é tudo o que fica implícito.
>
> **O primeiro implícito é a orientação da matriz.** A tabela desta seção põe as previsões nas **linhas**; o `confusion_matrix` do `scikit-learn` põe os valores **reais** nas linhas e as previsões nas colunas. É a transposta:
>
> ```python
> confusion_matrix(y_teste, previsto)
> # [[981070   4930]        linha 0 = sem leucemia (real)
> #  [ 13930     70]]       linha 1 = com leucemia (real)
>
> confusion_matrix(y_teste, previsto).ravel()
> # [981070   4930  13930     70]   ->  tn, fp, fn, tp
> ```
>
> Repare na ordem que o `.ravel()` produz: `tn, fp, fn, tp`. É exatamente o **inverso** da ordem em que as nossas funções recebem os argumentos — `precision(tp, fp, fn, tn)`. Desempacotar uma na outra sem olhar produz um número plausível e errado, sem erro nenhum: o código roda, o resultado é um float entre 0 e 1, e nada avisa. É a mesma família de armadilha que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) apontou ao embaralhar `xs` e `ys` separadamente — o programa não reclama, e o número que ele devolve não significa o que você acha que significa.
>
> **O segundo implícito é qual classe é a positiva.** As nossas funções não têm essa dúvida: `tp` é o primeiro argumento e você decidiu o que colocar ali. As da biblioteca têm `pos_label=1` e `average='binary'` por padrão, e daí saem dois comportamentos diferentes:
>
> - Se os rótulos forem texto, ela recusa: `ValueError: pos_label=1 is not a valid label. It should be one of ['ham' 'spam']`. Um erro barulhento, que se conserta passando `pos_label='spam'`.
> - Se os rótulos forem `0` e `1`, ela **não pergunta nada** e mede a classe `1`. Quando a classe rara é a `0` — "não pagou", "não converteu", "peça defeituosa" codificada como zero —, você recebe a precisão e a revocação da classe errada, com aparência perfeita de resposta certa. Nesse caso, o número que interessa vem de `precision_score(y_teste, previsto, pos_label=0)`.
>
> **O terceiro implícito é a divisão por zero, e ele é o mais interessante.** Um classificador que nunca prevê "positivo" tem `tp + fp == 0`, e a nossa `precision` estoura com `ZeroDivisionError` — o [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/05-usando-o-modelo.html) mostra isso acontecendo com dados reais. A biblioteca não estoura:
>
> ```python
> precision_score([1, 0, 1, 0], [0, 0, 0, 0])
> # UndefinedMetricWarning: Precision is ill-defined and being set to 0.0
> # due to no predicted samples. Use `zero_division` parameter to control
> # this behavior.
> # 0.0
> ```
>
> O padrão é `zero_division="warn"`, que devolve `0.0` e emite um aviso — e aviso, ao contrário de exceção, não interrompe nada. Num laço que testa vinte modelos e imprime uma tabela no fim, essa `0.0` entra na tabela como se fosse uma medição, indistinguível de um modelo que aponta positivos e erra todos. **São duas situações diferentes**: um modelo com precisão 0,0 aponta e erra; um modelo com precisão indefinida não aponta nada. O `ZeroDivisionError` do nosso código distingue os dois casos ao preço de derrubar o programa; o `0.0` da biblioteca não os distingue, e não derruba nada.
>
> Os valores aceitos são `"warn"`, `0.0`, `1.0` e `np.nan`. O último é o mais honesto para uma tabela comparativa: `nan` propaga, aparece como buraco em vez de nota baixa, e é excluído das médias.
>
> Por fim, `classification_report(y_teste, previsto)` devolve precisão, revocação, F1 e o número de exemplos (*support*) **para cada classe**, mais as médias macro e ponderada. É a correção direta do hábito de olhar só um lado, e o formato em que essas métricas normalmente são reportadas.

## O Compromisso Viés-Variância

> **📌 Nota**
>
> Esta seção corresponde a *The Bias-Variance Tradeoff*, do capítulo 11 de Grus (2019).

Outra forma de pensar no problema do overfitting é como um compromisso entre **viés** e **variância**.

Se as duas palavras soam familiares, é porque a [seção 3.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap03/03-graficos-de-linhas.html) já desenhou a figura deste compromisso — três curvas e um U — para demonstrar gráficos de linhas. Lá ela era forma; aqui é o conceito, e vale reabrir aquela figura depois de ler esta seção.

> **🔷 Conceito**
>
> Os dois são medidas do que aconteceria se você **retreinasse o seu modelo muitas vezes**, sobre conjuntos de treino diferentes, tirados da mesma população.
>
> **Viés** é o quanto o modelo erra sistematicamente, em qualquer conjunto de treino.
> **Variância** é o quanto o modelo *muda* de um conjunto de treino para outro.
>
> Repare que nenhuma das duas definições fala de um único ajuste. Elas falam de uma população de ajustes — e é por isso que não dá para medir viés e variância olhando um modelo treinado uma vez só.

Voltando ao gráfico da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html): o modelo de grau 0 vai errar bastante para praticamente qualquer conjunto de treino tirado daquela população, o que significa que ele tem **viés alto**. Em compensação, dois conjuntos de treino escolhidos ao acaso dariam modelos bem parecidos — a média de duas amostras da mesma população é parecida —, então ele tem **variância baixa**. Viés alto e variância baixa correspondem tipicamente a underfitting.

Já o modelo de grau 9 ajustou o conjunto de treino perfeitamente. Ele tem **viés muito baixo** e **variância muito alta**, porque dois conjuntos de treino diferentes produziriam modelos completamente diferentes. Isso corresponde a overfitting.

Dá para ver isso diretamente. O gráfico abaixo repete o experimento dez vezes, cada vez com dez pontos novos da mesma população, e desenha o modelo ajustado a cada uma:

In [ ]:
# Figura: Dez conjuntos de treino diferentes, da mesma população. À esquerda, grau 1; à direita, grau 9.
import numpy as np
from matplotlib import pyplot as plt

rng = np.random.default_rng(7)

def amostra(rng, n=10):
    xs = 2.5 + 0.65 * np.arange(n)
    ys = 2 * xs + rng.normal(0, 1.2, size=xs.shape)
    return xs, ys

def reta(xs, ys):
    dx = xs - xs.mean()
    dy = ys - ys.mean()
    slope = (dx * dy).sum() / (dx ** 2).sum()
    return ys.mean() - slope * xs.mean(), slope

def lagrange(x, xs, ys):
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for i in range(len(xs)):
        outros = np.arange(len(xs)) != i
        # x[:, None] põe a grade em coluna (301, 1) para casar com xs[outros] (9,)
        base = np.prod((x[:, None] - xs[outros]) / (xs[i] - xs[outros]), axis=1)
        total += ys[i] * base
    return total

grade = np.linspace(2.5, 8.35, 301)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for _ in range(10):
    xs, ys = amostra(rng)
    b, m = reta(xs, ys)
    ax1.plot(grade, b + m * grade, color="#2e7d32", alpha=0.6)
    ax2.plot(grade, lagrange(grade, xs, ys), color="#c00000", alpha=0.6)

ax1.set_title("grau 1 — variância baixa")
ax2.set_title("grau 9 — variância alta")
ax1.set_ylabel("y")
for ax in (ax1, ax2):
    ax.plot(grade, 2 * grade, "k--", lw=1, label="relação verdadeira")
    ax.set_ylim(0, 22)
    ax.set_xlabel("x")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

À esquerda, as dez retas quase se sobrepõem: o modelo é estável, mesmo que nenhuma delas acerte a relação verdadeira em cheio. À direita, os dez polinômios de grau 9 vão para todo lado — cada conjunto de treino produz um modelo essencialmente diferente, embora cada um deles acerte os seus próprios dez pontos perfeitamente.

### O que fazer com o diagnóstico

Pensar nos problemas do modelo dessa forma ajuda a decidir o que tentar quando ele não funciona bem.

**Se o seu modelo tem viés alto** — o que significa que ele vai mal até nos dados de treino —, uma coisa a tentar é **acrescentar atributos**. Sair do modelo de grau 0 para o de grau 1, na seção anterior, foi uma melhora grande.

**Se o seu modelo tem variância alta**, você pode do mesmo modo **remover atributos**. Mas há outra saída: **obter mais dados**, quando isso é possível.

O gráfico abaixo mostra o efeito de mais dados sobre a variância. É o mesmo experimento de antes, com dez amostras independentes, mas variando o tamanho de cada amostra:

In [ ]:
# Figura: Reduzindo a variância com mais dados: o mesmo modelo, ajustado a amostras de 10, 100 e 1000 pontos
rng = np.random.default_rng(7)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)

for ax, n in zip(axes, [10, 100, 1000]):
    for _ in range(10):
        xs = 2.5 + 5.85 * rng.random(n)
        ys = 2 * xs + rng.normal(0, 1.2, size=xs.shape)
        b, m = reta(xs, ys)
        ax.plot(grade, b + m * grade, color="#4472c4", alpha=0.6)
    ax.plot(grade, 2 * grade, "k--", lw=1)
    ax.set_title(f"N = {n}")
    ax.set_ylim(2, 20)
    ax.set_xlabel("x")

axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

Com dez pontos, as retas ajustadas se espalham visivelmente. Com mil, elas praticamente coincidem com a relação verdadeira. **Mantendo a complexidade do modelo constante, quanto mais dados você tem, mais difícil é sofrer overfitting.**

O efeito é ainda mais dramático em modelos complexos: o polinômio de grau 9, que com dez pontos se descontrola, com cem já melhora bastante e com mil fica muito parecido com o modelo de grau 1.

> **⚠️ Atenção — Mais dados não conserta viés**
>
> Por outro lado, mais dados **não** ajudam com viés. Se o seu modelo não usa atributos suficientes para capturar as regularidades dos dados, jogar mais dados nele não vai adiantar.
>
> Um modelo de grau 0 ajustado a um milhão de pontos continua sendo uma reta horizontal. Ele vai ser uma reta horizontal muito bem estimada, com a altura exata da média da população — e vai continuar sem descrever nada.
>
> Essa assimetria é útil na prática: "coletar mais dados" é uma resposta cara e frequentemente proposta. Ela resolve variância. Se o problema for viés, é dinheiro gasto sem efeito.

> **💡 Dica — Na prática: `learning_curve` e `validation_curve`**
>
> Os dois gráficos desta seção têm, cada um, uma função correspondente em `sklearn.model_selection` — e a correspondência é exata, não aproximada.
>
> Os números nos comentários abaixo saem de uma árvore de decisão sobre o Iris — o mesmo conjunto do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) — e são as médias sobre as cinco dobras da validação cruzada.
>
> O segundo gráfico varia o **tamanho da amostra** com o modelo fixo. É o `learning_curve`:
>
> ```python
> from sklearn.model_selection import learning_curve
> from sklearn.tree import DecisionTreeClassifier
>
> tamanhos, treino, teste = learning_curve(DecisionTreeClassifier(random_state=0),
>                                          X, y, shuffle=True, random_state=0)
> # tamanhos      [ 12  39  66  93 120]
> # treino.mean   [1.    1.    1.    1.    1.   ]
> # teste.mean    [0.787 0.933 0.953 0.94  0.96 ]
> ```
>
> A comparação entre os modelos de grau 1 e grau 9 varia a **complexidade** com a amostra fixa. É o `validation_curve`, que percorre os valores de um hiperparâmetro:
>
> ```python
> from sklearn.model_selection import validation_curve
>
> treino, teste = validation_curve(DecisionTreeClassifier(random_state=0), X, y,
>                                  param_name='max_depth',
>                                  param_range=[1, 2, 3, 5, 10])
> # treino.mean   [0.667 0.962 0.973 0.998 1.   ]
> # teste.mean    [0.667 0.933 0.96  0.96  0.96 ]
> ```
>
> As duas devolvem as pontuações de treino e de teste em arrays de forma `(n_pontos, n_dobras)` — uma linha por ponto do eixo horizontal, uma coluna por dobra da validação cruzada; o `learning_curve` devolve ainda os tamanhos usados. E o que se lê nesses arrays é o vocabulário desta seção. Na saída do `validation_curve` acima, com `max_depth=1` treino e teste empatam em 0,667 — o modelo vai mal nos dois, que é **viés alto**. Conforme a profundidade cresce, o treino sobe até 1,0 e o teste estaciona em 0,96: a curva de treino continua subindo e a de teste não a acompanha, e essa **distância entre as duas curvas é a leitura visual da variância**. Na saída do `learning_curve`, o treino é 1,0 desde a primeira amostra de 12 pontos — uma árvore sem profundidade máxima ajusta perfeitamente qualquer conjunto de treino, que é a afirmação "erro baixo no treino não é evidência de nada" impressa como número.
>
> **O `shuffle=True` acima não é decoração.** O padrão de `learning_curve` é `shuffle=False`, e as amostras menores são então tomadas do **começo** da partição de treino de cada dobra. Sobre um arquivo ordenado por classe — o `dados/iris.data` do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) é exatamente assim —, os primeiros pontos do eixo enxergam uma classe só. Com `shuffle=False` a mesma curva começa em 0,333 e fica lá por dois pontos, antes de subir; com `shuffle=True`, começa em 0,787. É a mesma armadilha do `KFold` sem embaralhamento, descrita na [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html), e a curva enganosa não vem acompanhada de aviso nenhum.
>
> **E há o que a biblioteca não oferece.** Não existe, em lugar nenhum do `scikit-learn`, uma função que decomponha o erro em viés e variância. Não é um esquecimento: a definição desta seção exige **retreinar o modelo sobre muitas amostras independentes da mesma população**, e com dados reais você tem uma amostra só. Foi por isso que os gráficos daqui precisaram de dados simulados, em que a população é conhecida e dá para sortear dez amostras dela.
>
> O que o `learning_curve` e o `validation_curve` entregam é um **diagnóstico por procuração**: duas curvas cuja altura e cuja distância sugerem viés e variância sem medir nenhum dos dois. É informação útil, e é o que existe. Mas quando alguém diz que "o modelo tem variância alta" olhando uma dessas figuras, está lendo um sintoma — e vale saber que o número por trás dele não foi calculado.

## Extração e Seleção de Atributos

> **📌 Nota**
>
> Esta seção corresponde a *Feature Extraction and Selection*, do capítulo 11 de Grus (2019).

Como já foi mencionado, quando os seus dados não têm atributos suficientes, o modelo tende a sofrer underfitting. E quando têm atributos demais, é fácil sofrer overfitting. Mas o que são atributos, afinal, e de onde eles vêm?

> **🔷 Conceito**
>
> **Atributos** (*features*) são quaisquer entradas que fornecemos ao nosso modelo.

No caso mais simples, os atributos simplesmente lhe são dados. Se você quer prever o salário de alguém a partir dos anos de experiência, então anos de experiência é o único atributo que você tem. (Embora, como se viu na [seção sobre overfitting](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html), você possa considerar acrescentar os anos de experiência ao quadrado, ao cubo, e assim por diante, se isso ajudar a construir um modelo melhor.)

As coisas ficam mais interessantes conforme os dados ficam mais complicados. Imagine construir um filtro de spam para prever se um e-mail é lixo ou não. A maioria dos modelos não sabe o que fazer com um e-mail bruto, que é apenas uma coleção de texto. Você vai ter que **extrair atributos**. Por exemplo:

- O e-mail contém a palavra *Viagra*?
- Quantas vezes a letra *d* aparece?
- Qual era o domínio do remetente?

A resposta à primeira pergunta é simplesmente sim ou não, o que tipicamente codificamos como 1 ou 0. A segunda é um número. E a terceira é uma escolha dentro de um conjunto discreto de opções.

### Três tipos de atributo, e o que cada um permite

Quase sempre, os atributos que extraímos dos dados caem em uma dessas três categorias. E — este é o ponto que costuma passar despercebido — **os tipos de atributo que você tem restringem os tipos de modelo que você pode usar**:

- O classificador **Naive Bayes**, que construiremos no [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html), é adequado a atributos do tipo sim-ou-não, como o primeiro da lista acima.
- Os modelos de **regressão**, dos capítulos [11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html), [12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html) e [13](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap13/index.html), exigem atributos numéricos — o que pode incluir variáveis indicadoras, que valem 0 ou 1.
- E as **árvores de decisão** em geral lidam com dados numéricos ou categóricos — mas a que o [Capítulo 14](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap14/index.html) constrói, o ID3, só sabe perguntar "qual é o valor deste atributo?", o que trata toda coluna como categórica.

> **🟩 Exemplo**
>
> Essa restrição costuma ser apresentada como detalhe de implementação, mas ela inverte a ordem em que as decisões parecem acontecer.
>
> A sequência intuitiva é: escolho o modelo, depois preparo os dados para ele. A sequência real é frequentemente a oposta — o formato do dado que você conseguiu obter já eliminou metade dos modelos antes de você chegar a considerá-los.
>
> É mais uma razão pela qual o [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html), sobre obter dados, vem antes dos capítulos de modelo neste livro.

### Remover atributos também é trabalho

Embora no exemplo do filtro de spam a gente tenha procurado formas de *criar* atributos, às vezes o que se procura é o contrário: formas de **remover** atributos.

Por exemplo, as suas entradas podem ser vetores de várias centenas de números. Dependendo da situação, pode ser apropriado destilá-los a um punhado de dimensões importantes — é o que faz a redução de dimensionalidade, no [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html) — e usar apenas esse número pequeno de atributos. Ou pode ser apropriado usar uma técnica, como a regularização do [Capítulo 12](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/index.html), que penaliza modelos quanto mais atributos eles usarem.

> **📌 Nota**
>
> As duas abordagens atacam o mesmo problema por caminhos diferentes.
>
> A **redução de dimensionalidade** decide, antes de treinar, quais direções dos dados carregam informação, e joga o resto fora.
>
> A **regularização** deixa todos os atributos disponíveis, mas cobra um preço por usá-los — o modelo só mantém um atributo se ele pagar por si em desempenho.
>
> A primeira é uma decisão sobre os dados; a segunda, uma decisão sobre o modelo. E as duas são respostas ao mesmo diagnóstico da seção anterior: variância alta.

### Como escolher atributos

E como escolhemos os atributos? É aí que entra a combinação de **experiência** e **conhecimento do domínio**.

Se você já recebeu muitos e-mails, provavelmente tem alguma intuição de que a presença de certas palavras pode ser um bom indicador de spam. E provavelmente também tem a intuição de que o número de letras *d* não é um bom indicador. Mas, em geral, você vai ter que testar coisas diferentes — o que faz parte da diversão.

> **💡 Dica — Na prática: `sklearn.feature_selection`**
>
> Existe um módulo inteiro dedicado à metade *seleção* desta seção. Ele se divide em duas famílias, e a diferença entre as duas é o assunto deste callout.
>
> **A primeira família pontua cada coluna sozinha**, por uma estatística univariada, e fica com as melhores. `SelectKBest(score_func=f_classif, k=10)` é a porta de entrada; `f_classif` é uma ANOVA de uma via, `chi2` serve a contagens não negativas, `mutual_info_classif` a relações não lineares.
>
> **A segunda família treina um modelo e pergunta a ele.** `SelectFromModel(estimator)` lê os coeficientes ou as importâncias do modelo ajustado; `RFE(estimator, n_features_to_select=k)` vai além e treina repetidamente, descartando a pior coluna a cada rodada.
>
> A diferença não é de sofisticação — é de **o que cada uma consegue enxergar**. Uma estatística univariada olha uma coluna de cada vez, então uma coluna que só significa alguma coisa *em combinação com outra* é invisível para ela. O caso mínimo tem três colunas: `a` e `b` sorteadas em $\{0, 1\}$, `ruído` uniforme, e o rótulo sendo `a XOR b`.
>
> ```python
> import numpy as np
> from sklearn.ensemble import RandomForestClassifier
> from sklearn.feature_selection import SelectKBest, SelectFromModel, f_classif
>
> rng = np.random.default_rng(7)
> n = 400
> a     = rng.integers(0, 2, size=n)
> b     = rng.integers(0, 2, size=n)
> ruido = rng.random(n)
>
> y = a ^ b                                     # y só existe na combinação de a e b
> X = np.column_stack([a, b, ruido]).astype(float)
>
> SelectKBest(f_classif, k=2).fit(X, y)
> # .scores_                            [0.165, 1.695, 1.160]   (a, b, ruído)
> # .get_support()                      [False,  True,  True]
>
> SelectFromModel(RandomForestClassifier(random_state=0), max_features=2).fit(X, y)
> # .estimator_.feature_importances_    [0.524, 0.395, 0.081]
> # .get_support()                      [ True,  True, False]
> ```
>
> Olhe os `scores_`. A coluna `a` — que, junto com `b`, **determina** o rótulo — pontua 0,165; a coluna de puro ruído pontua 1,160, sete vezes mais. O `SelectKBest` então descarta `a`, uma das duas únicas colunas informativas, e mantém o ruído. O `SelectFromModel` com uma floresta acerta, porque a floresta usa `a` e `b` juntas dentro da mesma árvore e a importância registra isso.
>
> Os valores exatos mudam com o sorteio; a cegueira, não. Isolada, nem `a` nem `b` diz coisa alguma sobre `y` — a pontuação de cada uma é ruído amostral, e cai ora acima, ora abaixo da coluna que é ruído de verdade. Repetindo o experimento com 300 sementes diferentes, o `SelectKBest` ficou com as duas colunas informativas em 116 delas: pouco mais do que as 100 que sortear duas de três colunas daria.
>
> O XOR é o caso extremo, escolhido para deixar o efeito visível. Mas ele não é exótico: qualquer interação entre atributos — um limiar de renda que só vale para certa faixa etária, uma palavra que só indica spam quando o remetente é desconhecido — produz a mesma cegueira em grau menor.
>
> Três padrões que a biblioteca escolhe por você, e que vale conhecer:
>
> - `SelectKBest` vem com **`k=10`**, um número sem nenhuma relação com os seus dados. Se você tiver menos colunas que isso, ele não falha — emite `UserWarning: k=10 is greater than n_features=3. All the features will be returned.` e devolve tudo, ou seja, não seleciona nada.
> - `SelectFromModel` sem `threshold` corta pela **média das importâncias**. Quem decide quantas colunas sobram não é você: é a distribuição das importâncias. Passe `max_features` se quiser um número.
> - `chi2` recusa entrada negativa, com `ValueError: Input X must be non-negative.` — é uma estatística de contagem, e um atributo já padronizado não serve para ela.
>
> **E há uma armadilha maior, que é a [seção 8.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/03-overfitting-e-underfitting.html) de volta.** Selecionar atributos *é* treinar: a seleção lê os rótulos. Fazê-la sobre o conjunto inteiro, antes de dividir, vaza o teste para dentro da escolha. O tamanho do estrago é fácil de medir com dados que não têm sinal nenhum — 100 linhas, 2.000 colunas de ruído gaussiano, rótulo alternando 0 e 1:
>
> ```python
> from sklearn.model_selection import cross_val_score
> from sklearn.neighbors import KNeighborsClassifier
> from sklearn.pipeline import Pipeline
>
> X = rng.normal(0, 1, size=(100, 2000))   # 100 linhas, 2.000 colunas de puro ruído
> y = np.array([0, 1] * 50)
>
> # ERRADO: escolhe as 10 melhores olhando tudo, depois valida
> X_sel = SelectKBest(f_classif, k=10).fit_transform(X, y)
> cross_val_score(KNeighborsClassifier(5), X_sel, y).mean()   # 0.81
>
> # CERTO: a seleção é refeita dentro de cada dobra
> pipe = Pipeline([('sel', SelectKBest(f_classif, k=10)),
>                  ('knn', KNeighborsClassifier(5))])
> cross_val_score(pipe, X, y).mean()                          # 0.45
> ```
>
> **81% de acurácia sobre dados onde não existe nada para aprender.** Com 2.000 colunas de ruído e 100 linhas, algumas colunas se correlacionam com o rótulo por acaso; escolhê-las olhando todas as linhas as leva para dentro do modelo, e a validação cruzada seguinte já não tem como desfazer isso. O `Pipeline` conserta porque refaz a seleção dentro de cada dobra, usando só os dados de treino daquela dobra — e aí o número volta para perto de 0,5, que é a verdade.
>
> Por fim, o limite honesto: **nada disso faz extração.** Este módulo ordena colunas que você já tem; ele nunca inventa a coluna "contém a palavra *Viagra*". Aquela coluna saiu de alguém que já leu muito spam, e continua sendo o trabalho que esta seção descreve como experiência e conhecimento do domínio. A seleção automática é a parte fácil, e é a única que a biblioteca faz.

> **❗ Importante — Onde este capítulo deságua**
>
> Este capítulo não construiu nenhum modelo, e ainda assim é o que torna os próximos oito legíveis. Ele entregou três coisas:
>
> **Um critério.** Um modelo é julgado no que ele não viu. Erro baixo no treino não é evidência de nada, e é por isso que todo capítulo daqui em diante vai separar dados antes de anunciar um resultado.
>
> **Um vocabulário.** Acurácia, precisão, revocação e F1 medem coisas diferentes, e a escolha entre elas é uma escolha sobre qual erro custa mais caro — que é uma pergunta do domínio, não do algoritmo.
>
> **Um diagnóstico.** Quando um modelo vai mal, viés e variância apontam para ações opostas: mais atributos contra menos atributos, e mais dados resolvendo apenas um dos dois casos.
>
> O [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html), sobre k-vizinhos, é o primeiro a usar tudo isso — e se você já o leu, vale reler a matriz de confusão de lá agora sabendo o que ela mede.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 11 de Grus (2019) sugere:

- Continuar lendo — os capítulos seguintes tratam de famílias diferentes de modelos de aprendizado de máquina.
- O curso [Machine Learning](https://www.coursera.org/learn/machine-learning) do Coursera, o MOOC original da área, para um entendimento mais profundo dos fundamentos.
- *The Elements of Statistical Learning*, de Friedman, Tibshirani e Hastie (Hastie et al., 2009), um livro-texto canônico, [disponível gratuitamente](https://hastie.su.domains/ElemStatLearn/) — com o aviso do autor de que ele é *muito* matemático.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.